# 🎙️ Speech-to-Text & Summarization Backend (Pipeline Approach - Decoder-Only LLM)

This Google Colab notebook implements an automated meeting intelligence backend using the Hugging Face **`pipeline`** abstraction.

- **Hugging Face Authentication:** Integrated support for personal Access Tokens (allows loading gated models and avoids rate limits).
- **ASR Model:** `openai/whisper-small` (via `pipeline("automatic-speech-recognition")`)
- **Summarization Model:** `Qwen/Qwen2.5-1.5B-Instruct` (modern Decoder-Only Causal LLM via `pipeline("text-generation")`)
- **Chat Templating:** Automated Jinja chat templates via `tokenizer.apply_chat_template()`
- **API Gateway:** FastAPI exposed to the public internet via `pyngrok`

---

## 📋 Quickstart Instructions

1. **Enable GPU Acceleration**:
   - In the top menu, select **Runtime** > **Change runtime type**.
   - Set **Hardware accelerator** to **T4 GPU** and click **Save**.

2. **Install Dependencies**:
   - Run **Step 1** below to install all required packages.

3. **Configure Hugging Face & ngrok Tokens**:
   - **Hugging Face Token:** In **Step 2**, paste your token into `HF_TOKEN = "YOUR_HF_TOKEN_HERE"` (from [HF Token Settings](https://huggingface.co/settings/tokens)) or add it to Colab Secrets.
   - **ngrok Token:** In **Step 4**, paste your token into `NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"` (from [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)).

4. **Start the Backend Server**:
   - Run **Step 2**, **Step 3**, and **Step 4** sequentially.
   - The notebook authenticates, downloads models, mounts the FastAPI routes, and opens a public ngrok tunnel.

5. **Connect from Local Gradio Client**:
   - Copy the printed public ngrok URL (e.g., `https://xxxx-xx-xx-xx.ngrok-free.app`).
   - Paste it into the `Colab Backend URL` field in your local `client.py` application.
   - Keep this Colab tab active while processing audio files.

## Step 1: Install Required Dependencies

In [ ]:
# Install core machine learning, authentication, and API server packages
!pip install -q transformers accelerate soundfile librosa fastapi uvicorn pyngrok nest-asyncio python-multipart torch torchaudio huggingface_hub

## Step 2: Hugging Face Authentication & Pipeline Initialization

### 🔑 Hugging Face Authentication
Authenticating with your Hugging Face token grants access to your gated repositories (such as LLaMA or Gemma models) and lifts anonymous download rate limits.

You can provide your token in either of two ways:
1. Paste it directly into `HF_TOKEN = "YOUR_HF_TOKEN_HERE"` below.
2. Add a secret named `HF_TOKEN` in the Google Colab Secrets tab (key icon in the left sidebar).

In [ ]:
import torch
from huggingface_hub import login
from transformers import pipeline, AutoTokenizer

# ----------------------------------------------------------------------------
# 1. Hugging Face Authentication
# ----------------------------------------------------------------------------
# Retrieve from Colab Secrets if available, otherwise use manual token variable
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    # Paste your Hugging Face User Access Token below:
    # Get one from: https://huggingface.co/settings/tokens (Read role is sufficient)
    HF_TOKEN = "YOUR_HF_TOKEN_HERE"

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
    print("[✓] Successfully authenticated with Hugging Face Hub!")
    hf_auth = HF_TOKEN
else:
    print("[!] No Hugging Face token set. Proceeding with anonymous public access...")
    hf_auth = None

# Verify GPU device availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"[*] Compute device: {device} | Torch dtype: {torch_dtype}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")

# ----------------------------------------------------------------------------
# 2. Load Whisper ASR Pipeline
# ----------------------------------------------------------------------------
print("[*] Loading Whisper ASR pipeline ('openai/whisper-small')...")
asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-small",
    chunk_length_s=30,
    stride_length_s=5,
    device=device,
    torch_dtype=torch_dtype,
    return_timestamps=False,
    token=hf_auth
)
print("[✓] ASR pipeline loaded!")

# ----------------------------------------------------------------------------
# 3. Load Decoder-Only LLM Text Generation Pipeline
# ----------------------------------------------------------------------------
llm_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"[*] Loading Decoder-Only LLM pipeline ('{llm_model_id}')...")
summarizer_pipe = pipeline(
    task="text-generation",
    model=llm_model_id,
    device=device,
    torch_dtype=torch_dtype,
    model_kwargs={"low_cpu_mem_usage": True},
    token=hf_auth
)
print("[✓] Decoder-Only LLM summarization pipeline loaded!")

## Step 3: Define FastAPI Application & Inference Logic

### Why Decoder-Only Models Use `apply_chat_template`:
A decoder-only model perceives all text as one single sequence of tokens. To indicate roles (system instructions vs. user input vs. assistant answer), we use `tokenizer.apply_chat_template()` with `add_generation_prompt=True` to format the conversation into the model's native format.

In [ ]:
import os
import time
import tempfile
import librosa
from typing import Dict, Any, List
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="Speech-to-Text & Summarization Backend (Pipeline - Decoder-Only)",
    description="Production backend using Whisper ASR and Qwen2.5 Decoder-Only LLM for meeting summarization",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health_check() -> Dict[str, Any]:
    """Diagnostics endpoint used by the Gradio client."""
    return {
        "status": "ok",
        "mode": "pipeline",
        "model_asr": "openai/whisper-small",
        "model_summarizer": "Qwen/Qwen2.5-1.5B-Instruct (Decoder-Only)",
        "device": str(device)
    }

def summarize_with_decoder_pipeline(text: str, min_length: int = 30, max_length: int = 150) -> str:
    """
    Summarizes meeting transcript using a Decoder-Only instruction-tuned model.
    Applies chat templating to properly demarcate system and user dialogue turns.
    """
    if not text.strip():
        return "No speech detected to summarize."

    messages: List[Dict[str, str]] = [
        {
            "role": "system",
            "content": (
                "You are an expert executive meeting assistant. Provide a concise, clear, and "
                "objective summary of the meeting transcription, highlighting key discussion points, "
                "decisions, and action items. Do not repeat the entire transcript."
            )
        },
        {
            "role": "user",
            "content": f"Please summarize the following meeting transcription:\n\n{text}"
        }
    ]

    # 1. Format messages with the model's native chat template
    # add_generation_prompt=True appends the assistant role marker to prompt generation
    prompt = summarizer_pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 2. Generate summary via pipeline with return_full_text=False
    # return_full_text=False ensures the pipeline returns ONLY newly generated response tokens
    output = summarizer_pipe(
        prompt,
        max_new_tokens=max_length,
        min_new_tokens=min(min_length, max_length),
        do_sample=False,
        return_full_text=False,
        pad_token_id=summarizer_pipe.tokenizer.pad_token_id or summarizer_pipe.tokenizer.eos_token_id
    )

    generated_summary = output[0]["generated_text"].strip()
    return generated_summary

@app.post("/process")
async def process_audio(
    file: UploadFile = File(...),
    min_length: int = Form(30),
    max_length: int = Form(150)
) -> Dict[str, Any]:
    """
    Receives an uploaded audio file, executes ASR transcription and decoder-only summarization,
    and returns a structured JSON payload with execution telemetry.
    """
    temp_audio_path = None
    try:
        # 1. Save uploaded bytes to a temporary audio file
        suffix = os.path.splitext(file.filename or "audio.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp_audio:
            content = await file.read()
            temp_audio.write(content)
            temp_audio_path = temp_audio.name

        # 2. Extract audio duration
        try:
            duration = float(librosa.get_duration(path=temp_audio_path))
        except Exception:
            duration = 0.0

        # 3. Transcribe via Whisper Pipeline
        t_asr_start = time.time()
        asr_result = asr_pipe(temp_audio_path)
        transcription = asr_result.get("text", "").strip()
        asr_time = round(time.time() - t_asr_start, 2)

        # 4. Summarize via Decoder-Only LLM Pipeline
        t_sum_start = time.time()
        if transcription:
            summary = summarize_with_decoder_pipeline(
                transcription,
                min_length=min_length,
                max_length=max_length
            )
        else:
            summary = "No spoken content was detected in the audio file."
        sum_time = round(time.time() - t_sum_start, 2)

        total_time = round(asr_time + sum_time, 2)

        return {
            "status": "success",
            "transcription": transcription,
            "summary": summary,
            "audio_duration_seconds": round(duration, 2),
            "metrics": {
                "transcription_time_seconds": asr_time,
                "summarization_time_seconds": sum_time,
                "total_time_seconds": total_time
            }
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Processing error: {str(exc)}")
    finally:
        if temp_audio_path and os.path.exists(temp_audio_path):
            try:
                os.remove(temp_audio_path)
            except Exception:
                pass

## Step 4: Configure ngrok Tunnel & Launch Uvicorn Server

Paste your ngrok token below and run the cell. Once the public URL appears, copy and paste it into `client.py`.

In [ ]:
import uvicorn
from pyngrok import ngrok
import nest_asyncio

# 1. Configure ngrok Authentication Token
# Replace with your personal token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Open Public Tunnel to Port 8000
public_url = ngrok.connect(8000).public_url

print("=" * 65)
print("🚀 Google Colab Pipeline Backend (Decoder-Only LLM) is ONLINE!")
print(f"[*] Public ngrok URL: {public_url}")
print(f"[*] Health check:     {public_url}/health")
print(f"[*] Process endpoint: {public_url}/process")
print("=" * 65)
print("👉 Copy the Public ngrok URL above and paste it into client.py")
print("=" * 65)

# 3. Run FastAPI ASGI Server
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()